# Thesis Experiment — Pitch–Timbre Coupling Reproduction

Per-descriptor, per-condition coupling analysis on $\log_2 f_0$ and its
derivatives. Conditions: **reference**, **DDSP**, **baseline (WORLD/SMS)**.

For each spectral descriptor $d_i$ and condition $c$, fit
$$d_i \;=\; \beta_0 + \beta_1 \log_2 f_0 + \beta_2 \dot{p} + \beta_3 \ddot{p} + \varepsilon,$$
then report within-condition fit metrics, coupling-comparison metrics
against the reference, and cross-prediction error from the reference
model onto each synthesized condition.

## Setup

In [28]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation.pitch_metrics import PitchMetrics

print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


## Configuration

In [29]:
PITCH_TRANSFORM = "log2"
SR              = 16_000
FRAME_WIDTH     = 0.25
OVERLAP         = 0.0
CONFIDENCE_THRESHOLD = 0.50
MEDIAN_WINDOW   = 5

DESCRIPTORS = [
    "spectral_centroid",
    "spectral_crest",
    "spectral_decrease",
    "spectral_flatness",
    "spectral_roll_off",
    "spectral_skewness",
    "spectral_spread",
]

# Input parquets. `ddsp` uses the reverb-trained DDSP run; `ddsp_noreverb`
# uses the no-reverb run. Both are compared against the same reference.
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "normalized"
REF_PARQUET = FEATURE_DIR / "bach_features_norm.parquet"
CONDITION_PARQUETS = {
    "reference":     REF_PARQUET,
    "ddsp":          FEATURE_DIR / "transfered_features_norm.parquet",
    "ddsp_noreverb": FEATURE_DIR / "transfered_noreverb_features_norm.parquet",
    "baseline":      FEATURE_DIR / "baseline_features_norm.parquet",
}
CONDITIONS     = list(CONDITION_PARQUETS.keys())
SYNTH_CONDITIONS = [c for c in CONDITIONS if c != "reference"]

OUT_DIR = PROJECT_ROOT / "artifacts" / "evaluation" / "thesis_experiment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

dt = FRAME_WIDTH * (1.0 - OVERLAP)
print(f"PITCH_TRANSFORM = {PITCH_TRANSFORM!r}")
print(f"dt = {dt:.4f} s")
for c, p in CONDITION_PARQUETS.items():
    print(f"  {c:<14s}: {p.name}  (exists: {p.exists()})")


PITCH_TRANSFORM = 'log2'
dt = 0.2500 s
  reference     : bach_features_norm.parquet  (exists: True)
  ddsp          : transfered_features_norm.parquet  (exists: True)
  ddsp_noreverb : transfered_noreverb_features_norm.parquet  (exists: True)
  baseline      : baseline_features_norm.parquet  (exists: True)


## 1. Prepare features

Load each condition's parquet, mask unvoiced frames by CREPE confidence,
and build the common design matrix $[1,\ \log_2 f_0,\ \dot{p},\ \ddot{p}]$
via `PitchMetrics.build_design`. Unvoiced / NaN frames are recorded in the
voiced mask and dropped at fit time.

In [30]:
def load_condition(parquet_path: Path) -> dict:
    df = pd.read_parquet(parquet_path)
    missing = [c for c in DESCRIPTORS + ["f0_hz", "f0_confidence"] if c not in df.columns]
    if missing:
        raise ValueError(f"{parquet_path.name}: missing columns {missing}")

    f0 = df["f0_hz"].to_numpy(dtype=np.float64)
    f0[df["f0_confidence"].to_numpy() < CONFIDENCE_THRESHOLD] = np.nan
    features_2d = df[DESCRIPTORS].to_numpy(dtype=np.float64)

    # Diagnostic voiced count: same definition as coupling_analysis.ipynb so
    # the two notebooks\' printouts line up. The actual fit rebuilds its own
    # per-descriptor mask inside PitchMetrics.fit — this count does not
    # influence the regression.
    voiced = (f0 > 0) & np.isfinite(f0) & np.all(np.isfinite(features_2d), axis=1)

    return {
        "f0": f0,
        "features_2d": features_2d,
        "n_frames": int(len(f0)),
        "n_voiced": int(voiced.sum()),
    }

condition_data = {}
for cond, path in CONDITION_PARQUETS.items():
    condition_data[cond] = load_condition(path)
    cd = condition_data[cond]
    print(f"{cond:<10s} {cd['n_frames']:>7,} frames  {cd['n_voiced']:>7,} voiced")

COEF_NAMES = ("b0_intercept", "b1_log2f0", "b2_p_dot", "b3_p_ddot")
print(f"\nDesign columns: intercept + p, p_dot, p_ddot  (pitch_transform={PITCH_TRANSFORM!r})")
print(f"Coefficient labels: {COEF_NAMES}")


reference   50,347 frames   40,405 voiced
ddsp       106,805 frames   79,844 voiced
ddsp_noreverb 106,805 frames   91,434 voiced
baseline   106,856 frames   99,392 voiced

Design columns: intercept + p, p_dot, p_ddot  (pitch_transform='log2')
Coefficient labels: ('b0_intercept', 'b1_log2f0', 'b2_p_dot', 'b3_p_ddot')


## 2. Fit separate regressions per condition

One OLS per (condition, descriptor). Stored coefficients follow the layout
$[\hat{\beta}_0, \hat{\beta}_1, \hat{\beta}_2, \hat{\beta}_3]$ where
$\hat{\beta}_1$ is the $\log_2 f_0$ slope (primary coupling term).

In [31]:
def rmse_from_model(model: dict) -> float:
    n = model["n"]
    return float(np.sqrt(model["ss_res"] / n)) if n > 0 else float("nan")

condition_models = {}
for cond, cd in condition_data.items():
    condition_models[cond] = PitchMetrics.fit(
        cd["features_2d"], cd["f0"], DESCRIPTORS,
        regressors="B", method="ols",
        dt=dt, pitch_transform=PITCH_TRANSFORM,
        median_window=MEDIAN_WINDOW,
    )
    print(f"fit {cond}: {len(condition_models[cond])} descriptors")

fit reference: 7 descriptors
fit ddsp: 7 descriptors


fit ddsp_noreverb: 7 descriptors
fit baseline: 7 descriptors


## 3. Within-condition fit metrics

$R^2$ and RMSE per (condition, descriptor).

In [32]:
within_rows = []
for cond, models in condition_models.items():
    for desc in DESCRIPTORS:
        m = models[desc]
        within_rows.append({
            "condition":  cond,
            "descriptor": desc,
            "n_voiced":   m["n"],
            "R2":         m["r2"],
            "adj_R2":     m["adj_r2"],
            "RMSE":       rmse_from_model(m),
        })

df_within = pd.DataFrame(within_rows)
print("Within-condition fit metrics (long form):")
display(df_within.round(4))

print("\nR² by (descriptor, condition):")
display(df_within.pivot(index="descriptor", columns="condition", values="R2").round(4))

print("\nRMSE by (descriptor, condition):")
display(df_within.pivot(index="descriptor", columns="condition", values="RMSE").round(4))

Within-condition fit metrics (long form):


,condition,descriptor,n_voiced,R2,adj_R2,RMSE
0,reference,spectral_centroid,32973,0.3029,0.3029,0.8283
1,reference,spectral_crest,32973,0.0020,0.0019,1.0166
2,reference,spectral_decrease,32973,0.0037,0.0036,0.0674
3,reference,spectral_flatness,32973,0.0120,0.0119,0.9022
4,reference,spectral_roll_off,32973,0.2161,0.2160,0.8702
5,reference,spectral_skewness,32973,0.1280,0.1279,0.0859
6,reference,spectral_spread,32973,0.1202,0.1202,0.9129
7,ddsp,spectral_centroid,64319,0.2352,0.2352,0.8556
8,ddsp,spectral_crest,64319,0.0002,0.0001,0.4479
9,ddsp,spectral_decrease,64319,0.0014,0.0013,0.0394



R² by (descriptor, condition):


condition,baseline,ddsp,ddsp_noreverb,reference
descriptor,,,,
spectral_centroid,0.1023,0.2352,0.7015,0.3029
spectral_crest,0.1365,0.0002,0.0003,0.0020
spectral_decrease,0.1508,0.0014,0.0018,0.0037
spectral_flatness,0.0524,0.0051,0.1132,0.0120
spectral_roll_off,0.0679,0.0518,0.4120,0.2161
spectral_skewness,0.0352,0.1750,0.5979,0.1280
spectral_spread,0.0230,0.0008,0.2520,0.1202



RMSE by (descriptor, condition):


condition,baseline,ddsp,ddsp_noreverb,reference
descriptor,,,,
spectral_centroid,0.9337,0.8556,0.5474,0.8283
spectral_crest,0.4127,0.4479,0.4920,1.0166
spectral_decrease,0.0163,0.0394,0.0429,0.0674
spectral_flatness,1.4406,1.6740,1.2089,0.9022
spectral_roll_off,0.8380,1.0123,0.6628,0.8702
spectral_skewness,0.0723,0.0637,0.0609,0.0859
spectral_spread,0.9071,1.2989,0.7850,0.9129


## 4. Coupling comparison metrics

For each synthesized condition $c$ and each descriptor $d_i$:
* **Coefficient difference** $\Delta\hat{\beta}_k = \hat{\beta}_k^{(c)} - \hat{\beta}_k^{\text{ref}}$ for $k = 0,1,2,3$.
* **Slope ratio** $\hat{\beta}_1^{(c)} / \hat{\beta}_1^{\text{ref}}$ (the $\log_2 f_0$ coupling term).
* $\Delta R^2 = R^2_{\text{ref}} - R^2_{(c)}$.

In [33]:
ref_models = condition_models["reference"]

coupling_rows = []
for cond in SYNTH_CONDITIONS:
    for desc in DESCRIPTORS:
        m_ref   = ref_models[desc]
        m_synth = condition_models[cond][desc]

        b_ref   = np.asarray(m_ref["coeffs"],   dtype=np.float64)
        b_synth = np.asarray(m_synth["coeffs"], dtype=np.float64)
        diff    = b_synth - b_ref

        b1_ref = float(b_ref[1])
        slope_ratio = float(b_synth[1] / b1_ref) if abs(b1_ref) > 0 else float("nan")

        coupling_rows.append({
            "condition":   cond,
            "descriptor":  desc,
            "delta_b0":    float(diff[0]),
            "delta_b1":    float(diff[1]),
            "delta_b2":    float(diff[2]),
            "delta_b3":    float(diff[3]),
            "slope_ratio_b1": slope_ratio,
            "delta_R2":    float(m_ref["r2"] - m_synth["r2"]),
        })

df_coupling = pd.DataFrame(coupling_rows)
print("Coupling comparison metrics (long form):")
display(df_coupling.round(4))

print("\nSlope ratio b1^synth / b1^ref (log2 f0 coupling):")
display(df_coupling.pivot(index="descriptor", columns="condition", values="slope_ratio_b1").round(4))

print("\nΔR² = R²_ref - R²_synth:")
display(df_coupling.pivot(index="descriptor", columns="condition", values="delta_R2").round(4))

Coupling comparison metrics (long form):


,condition,descriptor,delta_b0,delta_b1,delta_b2,delta_b3,slope_ratio_b1,delta_R2
0,ddsp,spectral_centroid,1.4406,0.0046,0.0298,-0.0023,1.0052,0.0677
1,ddsp,spectral_crest,-1.6359,0.0614,0.0067,-0.0014,0.1401,0.0018
2,ddsp,spectral_decrease,-0.0523,0.0042,-0.0001,-0.0000,0.3819,0.0024
3,ddsp,spectral_flatness,7.2714,-0.3824,0.0287,0.0036,-1.3651,0.0069
4,ddsp,spectral_roll_off,4.2768,-0.3017,0.0074,-0.0031,0.5940,0.1643
5,ddsp,spectral_skewness,-0.0723,-0.0014,-0.0051,0.0000,1.0261,-0.0470
6,ddsp,spectral_spread,6.5818,-0.5522,-0.0030,-0.0052,-0.0098,0.1194
7,ddsp_noreverb,spectral_centroid,-1.9822,0.3513,0.0194,0.0067,1.3944,-0.3985
8,ddsp_noreverb,spectral_crest,-1.6660,0.0795,-0.0001,-0.0015,-0.1139,0.0017
9,ddsp_noreverb,spectral_decrease,-0.0474,0.0044,-0.0010,-0.0002,0.3395,0.0019



Slope ratio b1^synth / b1^ref (log2 f0 coupling):


condition,baseline,ddsp,ddsp_noreverb
descriptor,,,
spectral_centroid,0.4425,1.0052,1.3944
spectral_crest,-2.8751,0.1401,-0.1139
spectral_decrease,1.2733,0.3819,0.3395
spectral_flatness,-2.6211,-1.3651,3.9557
spectral_roll_off,0.3806,0.5940,1.1029
spectral_skewness,0.3217,1.0261,2.0448
spectral_spread,0.3188,-0.0098,1.2281



ΔR² = R²_ref - R²_synth:


condition,baseline,ddsp,ddsp_noreverb
descriptor,,,
spectral_centroid,0.2006,0.0677,-0.3985
spectral_crest,-0.1345,0.0018,0.0017
spectral_decrease,-0.1471,0.0024,0.0019
spectral_flatness,-0.0404,0.0069,-0.1012
spectral_roll_off,0.1482,0.1643,-0.1959
spectral_skewness,0.0927,-0.0470,-0.4700
spectral_spread,0.0973,0.1194,-0.1318


## 5. Cross-prediction MSE

Use the reference coefficients $\hat{\beta}^{\text{ref}}$ to predict
$\hat{d}_i$ on each synthesized condition's feature matrix (the synth
$f_0$ drives the design matrix). Compute MSE and $R^2$ between the
predicted and the *actual* synthesized descriptor values on voiced frames.

MSE is in the descriptor's own (normalized) units; NMSE normalizes by
the synth descriptor variance so 0 = perfect and 1 = as bad as predicting
the synth mean. $R^2 = 1 - \text{MSE}/\text{Var}(y^{(c)}_i)$ (can be
negative when the ref model predicts worse than the synth mean).

In [34]:
def cross_prediction_scores(
    features_2d: np.ndarray,
    f0: np.ndarray,
    ref_models: dict,
    feat_keys: list,
) -> dict:
    """Per-descriptor MSE, NMSE (vs synth var), R² of ref_models on synth.

    Raw MSE reuses PitchMetrics.coupling_error. R² and NMSE use the *synth*
    descriptor variance (not the reference's) so they measure how much of
    the synthesized signal's variance the reference model explains.
    """
    mse_by_key = PitchMetrics.coupling_error(
        features_2d, f0, ref_models, feat_keys, normalize="none",
    )
    predicted = PitchMetrics.predict(f0, ref_models, feat_keys)
    out = {}
    for i, key in enumerate(feat_keys):
        if key not in mse_by_key:
            continue
        mask = np.isfinite(features_2d[:, i]) & np.isfinite(predicted[:, i])
        n = int(mask.sum())
        mse = mse_by_key[key]
        var = float(np.var(features_2d[mask, i], ddof=0)) if n else float("nan")
        if np.isfinite(var) and var > 0:
            nmse = mse / var
            r2 = 1.0 - nmse
        else:
            nmse = r2 = float("nan")
        out[key] = {"mse": mse, "nmse": nmse, "r2": r2, "n": n}
    return out

cross_rows = []
for cond in SYNTH_CONDITIONS:
    cd = condition_data[cond]
    scores = cross_prediction_scores(cd["features_2d"], cd["f0"],
                                     ref_models, DESCRIPTORS)
    for desc in DESCRIPTORS:
        s = scores[desc]
        cross_rows.append({
            "condition":  cond,
            "descriptor": desc,
            "n_voiced":   s["n"],
            "cross_MSE":  s["mse"],
            "cross_NMSE": s["nmse"],
            "cross_R2":   s["r2"],
        })

df_cross = pd.DataFrame(cross_rows)
print("Cross-prediction with reference model on synth features:")
display(df_cross.round(4))

print("\nCross MSE by (descriptor, condition):")
display(df_cross.pivot(index="descriptor", columns="condition", values="cross_MSE").round(4))

print("\nCross R² by (descriptor, condition):")
display(df_cross.pivot(index="descriptor", columns="condition", values="cross_R2").round(4))


Cross-prediction with reference model on synth features:


,condition,descriptor,n_voiced,cross_MSE,cross_NMSE,cross_R2
0,ddsp,spectral_centroid,64319,2.9247,3.0556,-2.0556
1,ddsp,spectral_crest,64319,1.4488,7.2194,-6.2194
2,ddsp,spectral_decrease,64319,0.0019,1.1908,-0.1908
3,ddsp,spectral_flatness,64319,19.1443,6.7967,-5.7967
4,ddsp,spectral_roll_off,64319,4.0270,3.7260,-2.7260
5,ddsp,spectral_skewness,64319,0.0112,2.2694,-1.2694
6,ddsp,spectral_spread,64319,5.4218,3.2109,-2.2109
7,ddsp_noreverb,spectral_centroid,81374,1.1470,1.1429,-0.1429
8,ddsp_noreverb,spectral_crest,81374,1.2774,5.2762,-4.2762
9,ddsp_noreverb,spectral_decrease,81374,0.0020,1.0700,-0.0700



Cross MSE by (descriptor, condition):


condition,baseline,ddsp,ddsp_noreverb
descriptor,,,
spectral_centroid,3.9978,2.9247,1.1470
spectral_crest,1.8930,1.4488,1.2774
spectral_decrease,0.0003,0.0019,0.0020
spectral_flatness,14.5319,19.1443,11.6271
spectral_roll_off,3.0320,4.0270,1.9452
spectral_skewness,0.0239,0.0112,0.0059
spectral_spread,3.1758,5.4218,2.2041



Cross R² by (descriptor, condition):


condition,baseline,ddsp,ddsp_noreverb
descriptor,,,
spectral_centroid,-3.1164,-2.0556,-0.1429
spectral_crest,-8.5971,-6.2194,-4.2762
spectral_decrease,-0.1086,-0.1908,-0.0700
spectral_flatness,-5.6351,-5.7967,-6.0556
spectral_roll_off,-3.0249,-2.7260,-1.6035
spectral_skewness,-3.4090,-1.2694,0.3630
spectral_spread,-2.7709,-2.2109,-1.6753


## 6. Aggregate summary table

Combine within-condition fit metrics, coupling comparison metrics, and
cross-prediction scores into one long-form table keyed by
(condition, descriptor).

In [35]:
# Coefficient table in long form so the final summary carries the raw
# estimates alongside the derived metrics.
coef_rows = []
for cond, models in condition_models.items():
    for desc in DESCRIPTORS:
        b = np.asarray(models[desc]["coeffs"], dtype=np.float64)
        coef_rows.append({
            "condition":  cond,
            "descriptor": desc,
            **{name: float(v) for name, v in zip(COEF_NAMES, b)},
        })
df_coef = pd.DataFrame(coef_rows)

summary = df_within.merge(df_coef, on=["condition", "descriptor"], how="left")
summary = summary.merge(
    df_coupling,
    on=["condition", "descriptor"], how="left",  # NaN for reference rows
)
summary = summary.merge(
    df_cross.drop(columns="n_voiced"),
    on=["condition", "descriptor"], how="left",  # NaN for reference rows
)

# Stable ordering: condition in config order, descriptor in config order.
summary["condition"]  = pd.Categorical(summary["condition"],  categories=CONDITIONS,  ordered=True)
summary["descriptor"] = pd.Categorical(summary["descriptor"], categories=DESCRIPTORS, ordered=True)
summary = summary.sort_values(["descriptor", "condition"]).reset_index(drop=True)

print(f"Summary table: {summary.shape[0]} rows × {summary.shape[1]} cols")
display(summary.round(4))

Summary table: 28 rows × 19 cols


,condition,descriptor,n_voiced,R2,adj_R2,RMSE,b0_intercept,b1_log2f0,b2_p_dot,b3_p_ddot,delta_b0,delta_b1,delta_b2,delta_b3,slope_ratio_b1,delta_R2,cross_MSE,cross_NMSE,cross_R2
0,reference,spectral_centroid,32973,0.3029,0.3029,0.8283,-7.8951,0.8906,0.0800,0.0042,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ddsp,spectral_centroid,64319,0.2352,0.2352,0.8556,-6.4544,0.8952,0.1098,0.0019,1.4406,0.0046,0.0298,-0.0023,1.0052,0.0677,2.9247,3.0556,-2.0556
2,ddsp_noreverb,spectral_centroid,81374,0.7015,0.7015,0.5474,-9.8772,1.2419,0.0994,0.0109,-1.9822,0.3513,0.0194,0.0067,1.3944,-0.3985,1.1470,1.1429,-0.1429
3,baseline,spectral_centroid,92581,0.1023,0.1023,0.9337,-2.0668,0.3941,0.0251,0.0042,5.8282,-0.4965,-0.0549,-0.0000,0.4425,0.2006,3.9978,4.1164,-3.1164
4,reference,spectral_crest,32973,0.0020,0.0019,1.0166,0.7061,-0.0714,-0.0120,0.0007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ddsp,spectral_crest,64319,0.0002,0.0001,0.4479,-0.9298,-0.0100,-0.0053,-0.0007,-1.6359,0.0614,0.0067,-0.0014,0.1401,0.0018,1.4488,7.2194,-6.2194
6,ddsp_noreverb,spectral_crest,81374,0.0003,0.0002,0.4920,-0.9598,0.0081,-0.0121,-0.0008,-1.6660,0.0795,-0.0001,-0.0015,-0.1139,0.0017,1.2774,5.2762,-4.2762
7,baseline,spectral_crest,92581,0.1365,0.1365,0.4127,-2.8744,0.2052,0.0052,0.0017,-3.5805,0.2765,0.0172,0.0010,-2.8751,-0.1345,1.8930,9.5971,-8.5971
8,reference,spectral_decrease,32973,0.0037,0.0036,0.0674,0.0733,-0.0067,-0.0009,-0.0001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ddsp,spectral_decrease,64319,0.0014,0.0013,0.0394,0.0210,-0.0026,-0.0010,-0.0001,-0.0523,0.0042,-0.0001,-0.0000,0.3819,0.0024,0.0019,1.1908,-0.1908


### Save artifacts

In [36]:
df_within.to_csv  (OUT_DIR / "within_condition_fit.csv",  index=False)
df_coef.to_csv    (OUT_DIR / "coefficients.csv",          index=False)
df_coupling.to_csv(OUT_DIR / "coupling_comparison.csv",   index=False)
df_cross.to_csv   (OUT_DIR / "cross_prediction.csv",      index=False)
summary.to_csv    (OUT_DIR / "summary_per_descriptor_per_condition.csv", index=False)

print(f"Saved 5 CSVs to {OUT_DIR}")

Saved 5 CSVs to /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/evaluation/thesis_experiment
